# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10` (Ablation Study A1 -> A4)

In [ ]:
# BƯỚC 1: KIỂM TRA PHẦN CỨNG & TỰ ĐỘNG ĐỒNG BỘ MÃ NGUỒN (PORTABLE ENV)
import os, sys, shutil
from pathlib import Path
import torch

# 1. Tự động nhận diện và đồng bộ mã nguồn trên Kaggle nếu chưa có folder
if Path("/kaggle").exists():
    cwd = Path("/kaggle/working")
    os.chdir(cwd)
    if not (cwd / "training_lab").exists() and not (cwd / "training").exists():
        print("⚡ [Auto-Sync] Đang tự động tải mã nguồn đầy đủ từ GitHub ducmanh-jr/Multi-Agent_RAG...")
        os.system("git clone https://github.com/ducmanh-jr/Multi-Agent_RAG.git /kaggle/working/temp_repo")
        if (cwd / "temp_repo" / "training_lab").exists():
            shutil.copytree(str(cwd / "temp_repo" / "training_lab"), str(cwd / "training_lab"), dirs_exist_ok=True)
            shutil.rmtree(str(cwd / "temp_repo"), ignore_errors=True)
            print("✅ Đã tự động đồng bộ trọn vẹn thư mục training_lab vào /kaggle/working!")
        elif (cwd / "temp_repo" / "training").exists():
            for item in (cwd / "temp_repo").iterdir():
                if item.name != ".git" and item.name != "temp_repo":
                    dest = cwd / item.name
                    if item.is_dir():
                        shutil.copytree(str(item), str(dest), dirs_exist_ok=True)
                    else:
                        shutil.copy2(str(item), str(dest))
            shutil.rmtree(str(cwd / "temp_repo"), ignore_errors=True)
            print("✅ Đã đồng bộ mã nguồn gốc vào /kaggle/working!")

    # Quét thêm nếu người dùng đính kèm qua Kaggle Dataset (/kaggle/input/)
    if Path("/kaggle/input").exists() and not (cwd / "training_lab").exists():
        for p in Path("/kaggle/input").glob("**/*"):
            if p.name == "training_lab" and p.is_dir():
                shutil.copytree(str(p), str(cwd / "training_lab"), dirs_exist_ok=True)
                print(f"✅ Đã tự động sao chép mã nguồn từ Dataset: {p}")
                break

cwd = Path.cwd()
if cwd.name == 'training_lab':
    os.chdir(cwd.parent)
    cwd = Path.cwd()
elif not (cwd / 'training_lab').exists() and (cwd.parent / 'training_lab').exists():
    os.chdir(cwd.parent)
    cwd = Path.cwd()

root_dir = cwd
lab_dir = cwd / 'training_lab' if (cwd / 'training_lab').exists() else cwd

sep = ';' if os.name == 'nt' else ':'
existing_pp = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = f"{root_dir}{sep}{lab_dir}" + (f"{sep}{existing_pp}" if existing_pp else "")
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

print(f"📂 Thư mục gốc dự án: {root_dir}")
print(f"📁 Thư mục training_lab tồn tại: {(root_dir / 'training_lab').exists() or (root_dir / 'training').exists()}")
print(f"⚡ Số GPU khả dụng: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU {i}: {props.name} ({props.total_memory / 1024**3:.2f} GB VRAM)")

os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes==0.44.1 \
    accelerate==1.0.1 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


In [ ]:
# BƯỚC 3: HUẤN LUYỆN CROSS-ENCODER RERANKER (RECIPE CoREB: LoRA r=16, alpha=16, nDCG@10)
import os, sys, shutil
from pathlib import Path

# Đảm bảo mã nguồn sẵn sàng 100% trước khi import
if not os.path.exists("training_lab") and not os.path.exists("training"):
    if Path("/kaggle").exists():
        print("⚡ [Auto-Sync] Đang tự động nạp mã nguồn từ GitHub ducmanh-jr/Multi-Agent_RAG...")
        os.system("git clone https://github.com/ducmanh-jr/Multi-Agent_RAG.git /kaggle/working/temp_repo")
        if os.path.exists("/kaggle/working/temp_repo/training_lab"):
            shutil.copytree("/kaggle/working/temp_repo/training_lab", "/kaggle/working/training_lab", dirs_exist_ok=True)
            shutil.rmtree("/kaggle/working/temp_repo", ignore_errors=True)

print('\n' + '='*80 + '\n🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES\n' + '='*80)
mod = "training_lab.training.reranker.train_reranker_lora" if os.path.exists("training_lab") else "training.reranker.train_reranker_lora"
dpath = "training_lab/data/triplets/massive_reranker_pairs.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_reranker_pairs.jsonl"
!python -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
# BƯỚC 4: HUẤN LUYỆN EMBEDDING (RECIPE MoAI: LoRA r=32, CachedMNRLoss GradCache)
import os, sys
print('\n' + '='*80 + '\n⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING VỚI GRADCACHE (EFFECTIVE BATCH = 64)\n' + '='*80)
mod = "training_lab.training.embedding.train_embedding_lora" if os.path.exists("training_lab") else "training.embedding.train_embedding_lora"
dpath = "training_lab/data/triplets/massive_embedding_triplets.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_embedding_triplets.jsonl"
!python -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
# BƯỚC 5: HUẤN LUYỆN AGENT 2 (DOCUMENT READER) TRÊN TẬP DOCLAYNET THẬT (10 NHÃN)
import os
print('\n' + '='*80 + '\n📄 [STAGE 3] HUẤN LUYỆN LAYOUTLMv3 FOCAL LOSS TRÊN BẢNG & CÔNG THỨC THẬT\n' + '='*80)
target = "training_lab/agent2_reader/train_agent2_t4x2.py" if os.path.exists("training_lab") else "agent2_reader/train_agent2_t4x2.py"
!python {target} --epochs 3


In [ ]:
# BƯỚC 6: CHẠY ABLATION STUDY THỰC NGHIỆM ĐỘC LẬP (A1 -> A4)
import os
print('\n' + '='*80 + '\n🏆 [STAGE 4] CHẠY ABLATION STUDY ĐỐI CHỨNG THEO CHUẨN TURKISH LEGAL RAG\n' + '='*80)
mod = "training_lab.evaluation.run_ablation_benchmark" if os.path.exists("training_lab") else "evaluation.run_ablation_benchmark"
!python -m {mod}


In [ ]:
# ==============================================================================
# BƯỚC 7: ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD, RGB BENCHMARK & TỰ SỬA SAI)
# Đánh giá 4 chỉ số cấp thế giới: Negative Rejection, Noise Robustness, Groundedness, Latency
# ==============================================================================
import os
bm_script = "training_lab/evaluation/world_class_benchmark.py" if os.path.exists("training_lab") else "evaluation/world_class_benchmark.py"
!python {bm_script}


In [ ]:
# BƯỚC 8: TỰ ĐỘNG ĐÓNG GÓI TRỌNG SỐ VỀ TAB KAGGLE OUTPUT (HOẶC LOCAL OUTPUTS)
import shutil
from pathlib import Path

base_out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
BUNDLE_DIR = base_out / 'AI20_Trained_Adapters'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

# Gom trọng số các adapter đã huấn luyện
search_dirs = [Path('training_lab'), Path('.')]
found_adapters = set()
for sdir in search_dirs:
    if sdir.exists():
        for a in sdir.glob('**/adapter_model.safetensors'):
            # Bỏ qua các checkpoint trung gian, chỉ lấy checkpoint xuất bản cuối cùng
            if 'checkpoint' in str(a).lower():
                continue
            found_adapters.add(a.resolve())

if not found_adapters:
    print('⚠️ [CẢNH BÁO] Chưa tìm thấy adapter_model.safetensors nào để gom.')

for a in found_adapters:
    # Tránh va chạm thư mục khi nhiều agent cùng lưu trong weights/
    if a.parent.name == "weights":
        agent_name = a.parent.parent.name
    elif a.parent.parent.name == "weights":
        agent_name = f"{a.parent.parent.parent.name}_{a.parent.name}"
    else:
        agent_name = a.parent.name
    dest = BUNDLE_DIR / agent_name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(a, dest / 'adapter_model.safetensors')
    # Sao chép toàn bộ config và tokenizer metadata nếu có
    for cfg_f in a.parent.glob('*.json'):
        shutil.copy2(cfg_f, dest / cfg_f.name)
    print(f'📦 Đã gom hoàn chỉnh adapter & tokenizer: {agent_name} -> {dest}')

OUT_ZIP = base_out / 'AI20_MultiAgent_Weights_DualT4.zip'
shutil.make_archive(str(OUT_ZIP).replace('.zip', ''), 'zip', str(BUNDLE_DIR))
print('=' * 80)
print(f'🎉 [THÀNH CÔNG] File zip đã sẵn sàng tải về tại: {OUT_ZIP}')
print('👉 Bạn có thể tải file này từ tab Output bên phải màn hình Kaggle!')
print('=' * 80)
